In [1]:
#The script uses Python code to run three-dimensional Google Mediapipe pose estimation for the body model and both hand models.
#
#Inputs:
# A path to your video file. The name of your video file will be used to define the variable [your_filename], which will be used to define output filenames.
#
#Outputs:
# [your_filename].avi: video file with overlaid pose estimation keypoints
# [your_filename]_body_outputs.csv: This file contains the 3D pose estimation coordinates (in camera image-scaled coordinates) for body keypoints.
# [your_filename]_hand_outputs.csv: This file contains the 3D pose estimation coordinates for both left and right hand keypoints.

#Import libraries
import os

# Set your paths here (run this cell before running the next cell).
input_video_path = r"C:\Users\BurchK\Documents\Mediapipe_Finger_Individuation_Code\fingerIndiv.MOV" #this is the name and location of the video that you want to track

base_path_name = input_video_path[0:-4:1] #define the base path name which will be used to define the location and name of output files and folders

output_video_path = base_path_name + "_tracked.avi" #this is the name and location of your desired output video that will have your tracked landmarks overlaid on top of the input video
hand_output_csv_path =  base_path_name + "_hand_outputs.csv" #this is the name and location of your desired output .csv file where you will save the frame-by-frame pixel hand coordinate data
body_output_csv_path = base_path_name + "_body_outputs.csv" #this is the name and location of your desired output .csv file where you will save the frame-by-frame pixel body coordinate data


In [2]:
# Run to this cell perform Google Mediapipe pose estimation. 
# Be sure that you have previously installed Google Mediapipe and OpenCV.
# Installation instructions for Google Mediapipe are here: https://google.github.io/mediapipe/getting_started/install.html

#Import libraries
import time
import mediapipe as mp
import cv2
import numpy as np

start_time = time.time()

print('If you want to stop the analysis at any time, press "q" for quit')

mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic
mp_drawing_styles = mp.solutions.drawing_styles

# Define landmarks and save as .csv files.
landmark_names = np.array(['L_WRIST_X','L_WRIST_Y','L_WRIST_Z','L_THUMB_1_X','L_THUMB_1_Y','L_THUMB_1_Z','L_THUMB_2_X','L_THUMB_2_Y','L_THUMB_2_Z','L_THUMB_3_X','L_THUMB_3_Y','L_THUMB_3_Z','L_THUMB_4_X','L_THUMB_4_Y','L_THUMB_4_Z','L_INDEX_1_X','L_INDEX_1_Y','L_INDEX_1_Z','L_INDEX_2_X','L_INDEX_2_Y','L_INDEX_2_Z','L_INDEX_3_X','L_INDEX_3_Y','L_INDEX_3_Z','L_INDEX_4_X','L_INDEX_4_Y','L_INDEX_4_Z','L_MIDDLE_1_X','L_MIDDLE_1_Y','L_MIDDLE_1_Z','L_MIDDLE_2_X','L_MIDDLE_2_Y','L_MIDDLE_2_Z','L_MIDDLE_3_X','L_MIDDLE_3_Y','L_MIDDLE_3_Z','L_MIDDLE_4_X','L_MIDDLE_4_Y','L_MIDDLE_4_Z','L_RING_1_X','L_RING_1_Y','L_RING_1_Z','L_RING_2_X','L_RING_2_Y','L_RING_2_Z','L_RING_3_X','L_RING_3_Y','L_RING_3_Z','L_RING_4_X','L_RING_4_Y','L_RING_4_Z','L_PINKY_1_X','L_PINKY_1_Y','L_PINKY_1_Z','L_PINKY_2_X','L_PINKY_2_Y','L_PINKY_2_Z','L_PINKY_3_X','L_PINKY_3_Y','L_PINKY_3_Z','L_PINKY_4_X','L_PINKY_4_Y','L_PINKY_4_Z','R_WRIST_X','R_WRIST_Y','R_WRIST_Z','R_THUMB_1_X','R_THUMB_1_Y','R_THUMB_1_Z','R_THUMB_2_X','R_THUMB_2_Y','R_THUMB_2_Z','R_THUMB_3_X','R_THUMB_3_Y','R_THUMB_3_Z','R_THUMB_4_X','R_THUMB_4_Y','R_THUMB_4_Z','R_INDEX_1_X','R_INDEX_1_Y','R_INDEX_1_Z','R_INDEX_2_X','R_INDEX_2_Y','R_INDEX_2_Z','R_INDEX_3_X','R_INDEX_3_Y','R_INDEX_3_Z','R_INDEX_4_X','R_INDEX_4_Y','R_INDEX_4_Z','R_MIDDLE_1_X','R_MIDDLE_1_Y','R_MIDDLE_1_Z','R_MIDDLE_2_X','R_MIDDLE_2_Y','R_MIDDLE_2_Z','R_MIDDLE_3_X','R_MIDDLE_3_Y','R_MIDDLE_3_Z','R_MIDDLE_4_X','R_MIDDLE_4_Y','R_MIDDLE_4_Z','R_RING_1_X','R_RING_1_Y','R_RING_1_Z','R_RING_2_X','R_RING_2_Y','R_RING_2_Z','R_RING_3_X','R_RING_3_Y','R_RING_3_Z','R_RING_4_X','R_RING_4_Y','R_RING_4_Z','R_PINKY_1_X','R_PINKY_1_Y','R_PINKY_1_Z','R_PINKY_2_X','R_PINKY_2_Y','R_PINKY_2_Z','R_PINKY_3_X','R_PINKY_3_Y','R_PINKY_3_Z','R_PINKY_4_X','R_PINKY_4_Y','R_PINKY_4_Z'])
len_landmarks = len(landmark_names)
landmark_names.shape = (1,len_landmarks)
np.savetxt(hand_output_csv_path,landmark_names,fmt='%s',delimiter=',')

rh_landmark_names = np.array(['R_WRIST_X','R_WRIST_Y','R_WRIST_Z','R_THUMB_1_X','R_THUMB_1_Y','R_THUMB_1_Z','R_THUMB_2_X','R_THUMB_2_Y','R_THUMB_2_Z','R_THUMB_3_X','R_THUMB_3_Y','R_THUMB_3_Z','R_THUMB_4_X','R_THUMB_4_Y','R_THUMB_4_Z','R_INDEX_1_X','R_INDEX_1_Y','R_INDEX_1_Z','R_INDEX_2_X','R_INDEX_2_Y','R_INDEX_2_Z','R_INDEX_3_X','R_INDEX_3_Y','R_INDEX_3_Z','R_INDEX_4_X','R_INDEX_4_Y','R_INDEX_4_Z','R_MIDDLE_1_X','R_MIDDLE_1_Y','R_MIDDLE_1_Z','R_MIDDLE_2_X','R_MIDDLE_2_Y','R_MIDDLE_2_Z','R_MIDDLE_3_X','R_MIDDLE_3_Y','R_MIDDLE_3_Z','R_MIDDLE_4_X','R_MIDDLE_4_Y','R_MIDDLE_4_Z','R_RING_1_X','R_RING_1_Y','R_RING_1_Z','R_RING_2_X','R_RING_2_Y','R_RING_2_Z','R_RING_3_X','R_RING_3_Y','R_RING_3_Z','R_RING_4_X','R_RING_4_Y','R_RING_4_Z','R_PINKY_1_X','R_PINKY_1_Y','R_PINKY_1_Z','R_PINKY_2_X','R_PINKY_2_Y','R_PINKY_2_Z','R_PINKY_3_X','R_PINKY_3_Y','R_PINKY_3_Z','R_PINKY_4_X','R_PINKY_4_Y','R_PINKY_4_Z'])
rh_len_landmarks = len(rh_landmark_names)

lh_landmark_names = np.array(['L_WRIST_X','L_WRIST_Y','L_WRIST_Z','L_THUMB_1_X','L_THUMB_1_Y','L_THUMB_1_Z','L_THUMB_2_X','L_THUMB_2_Y','L_THUMB_2_Z','L_THUMB_3_X','L_THUMB_3_Y','L_THUMB_3_Z','L_THUMB_4_X','L_THUMB_4_Y','L_THUMB_4_Z','L_INDEX_1_X','L_INDEX_1_Y','L_INDEX_1_Z','L_INDEX_2_X','L_INDEX_2_Y','L_INDEX_2_Z','L_INDEX_3_X','L_INDEX_3_Y','L_INDEX_3_Z','L_INDEX_4_X','L_INDEX_4_Y','L_INDEX_4_Z','L_MIDDLE_1_X','L_MIDDLE_1_Y','L_MIDDLE_1_Z','L_MIDDLE_2_X','L_MIDDLE_2_Y','L_MIDDLE_2_Z','L_MIDDLE_3_X','L_MIDDLE_3_Y','L_MIDDLE_3_Z','L_MIDDLE_4_X','L_MIDDLE_4_Y','L_MIDDLE_4_Z','L_RING_1_X','L_RING_1_Y','L_RING_1_Z','L_RING_2_X','L_RING_2_Y','L_RING_2_Z','L_RING_3_X','L_RING_3_Y','L_RING_3_Z','L_RING_4_X','L_RING_4_Y','L_RING_4_Z','L_PINKY_1_X','L_PINKY_1_Y','L_PINKY_1_Z','L_PINKY_2_X','L_PINKY_2_Y','L_PINKY_2_Z','L_PINKY_3_X','L_PINKY_3_Y','L_PINKY_3_Z','L_PINKY_4_X','L_PINKY_4_Y','L_PINKY_4_Z'])
lh_len_landmarks = len(rh_landmark_names)

body_landmark_names = np.array(['NOSE_X','NOSE_Y','NOSE_Z','LEFT_EYE_INNER_X','LEFT_EYE_INNER_Y','LEFT_EYE_INNER_Z','LEFT_EYE_X','LEFT_EYE_Y','LEFT_EYE_Z','LEFT_EYE_OUTER_X','LEFT_EYE_OUTER_Y','LEFT_EYE_OUTER_Z','RIGHT_EYE_INNER_X','RIGHT_EYE_INNER_Y','RIGHT_EYE_INNER_Z','RIGHT_EYE_X','RIGHT_EYE_Y','RIGHT_EYE_Z','RIGHT_EYE_OUTER_X','RIGHT_EYE_OUTER_Y','RIGHT_EYE_OUTER_Z','LEFT_EAR_X','LEFT_EAR_Y','LEFT_EAR_Z','RIGHT_EAR_X','RIGHT_EAR_Y','RIGHT_EAR_Z','MOUTH_LEFT_X','MOUTH_LEFT_Y','MOUTH_LEFT_Z','MOUTH_RIGHT_X','MOUTH_RIGHT_Y','MOUTH_RIGHT_Z','LEFT_SHOULDER_X','LEFT_SHOULDER_Y','LEFT_SHOULDER_Z','RIGHT_SHOULDER_X','RIGHT_SHOULDER_Y','RIGHT_SHOULDER_Z','LEFT_ELBOW_X','LEFT_ELBOW_Y','LEFT_ELBOW_Z','RIGHT_ELBOW_X','RIGHT_ELBOW_Y','RIGHT_ELBOW_Z','LEFT_WRIST_X','LEFT_WRIST_Y','LEFT_WRIST_Z','RIGHT_WRIST_X','RIGHT_WRIST_Y','RIGHT_WRIST_Z','LEFT_PINKY_X','LEFT_PINKY_Y','LEFT_PINKY_Z','RIGHT_PINKY_X','RIGHT_PINKY_Y','RIGHT_PINKY_Z','LEFT_INDEX_X','LEFT_INDEX_Y','LEFT_INDEX_Z','RIGHT_INDEX_X','RIGHT_INDEX_Y','RIGHT_INDEX_Z','LEFT_THUMB_X','LEFT_THUMB_Y','LEFT_THUMB_Z','RIGHT_THUMB_X','RIGHT_THUMB_Y','RIGHT_THUMB_Z','LEFT_HIP_X','LEFT_HIP_Y','LEFT_HIP_Z','RIGHT_HIP_X','RIGHT_HIP_Y','RIGHT_HIP_Z','LEFT_KNEE_X','LEFT_KNEE_Y','LEFT_KNEE_Z','RIGHT_KNEE_X','RIGHT_KNEE_Y','RIGHT_KNEE_Z','LEFT_ANKLE_X','LEFT_ANKLE_Y','LEFT_ANKLE_Z','RIGHT_ANKLE_X','RIGHT_ANKLE_Y','RIGHT_ANKLE_Z','LEFT_HEEL_X','LEFT_HEEL_Y','LEFT_HEEL_Z','RIGHT_HEEL_X','RIGHT_HEEL_Y','RIGHT_HEEL_Z','LEFT_FOOT_INDEX_X','LEFT_FOOT_INDEX_Y','LEFT_FOOT_INDEX_Z','RIGHT_FOOT_INDEX_X','RIGHT_FOOT_INDEX_Y','RIGHT_FOOT_INDEX_Z'])
body_len_landmarks = len(body_landmark_names)
body_landmark_names.shape = (1,body_len_landmarks)
np.savetxt(body_output_csv_path,body_landmark_names,fmt='%s',delimiter=',')

# Initialize data arrays.
rh_data = np.array([])
lh_data = np.array([])
body_data = np.array([])

# Set iteration counter.
iter = 0

# Bring in video, define video output.
cap = cv2.VideoCapture(input_video_path)
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc('M','J','P','G'), 30, (frame_width, frame_height))

# Run pose estimation.
with mp_holistic.Holistic(min_detection_confidence=0.5,min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print_iter = str(iter)
            print_analysis_time = str(time.time() - start_time)
            print("Tracking complete, " + print_iter + ' frames analyzed')
            print("This analysis took " + print_analysis_time + ' seconds')
            print("Your tracked output video has been saved to " + output_video_path)
            print("Your hand output .csv data file has been saved to " + hand_output_csv_path)
            print("Your body output .csv data file has been saved to " + body_output_csv_path)
            break
    
        # Convert the BGR image to RGB.
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = holistic.process(image)

        # Draw the annotations on the image.
        image_height, image_width, _ = image.shape
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Body/pose
        if results.pose_landmarks is None:
            body_data = np.concatenate((body_data,np.zeros((body_len_landmarks,),dtype=int)),axis=None)
        elif results.pose_landmarks:
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, mp_drawing.DrawingSpec(color=(200,200,200),thickness=2,circle_radius=1),mp_drawing.DrawingSpec(color=(255,255,255),thickness=2,circle_radius=1))
            for point in mp_holistic.PoseLandmark:
                normalized_pose_landmark = results.pose_landmarks.landmark[point]
                pixel_coordinates_pose_landmark = [normalized_pose_landmark.x, normalized_pose_landmark.y, normalized_pose_landmark.z]
                body_data=np.concatenate((body_data,pixel_coordinates_pose_landmark[0], pixel_coordinates_pose_landmark[1],pixel_coordinates_pose_landmark[2]),axis=None)        

        # Left hand
        if results.left_hand_landmarks is None:
            lh_data=np.concatenate((lh_data,np.zeros((lh_len_landmarks,),dtype=int)),axis=None)
        elif results.left_hand_landmarks:
            mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_drawing.DrawingSpec(color=(255,0,0),thickness=2,circle_radius=1),mp_drawing.DrawingSpec(color=(255,150,150),thickness=2,circle_radius=1))
            for point in mp_holistic.HandLandmark:
                normalized_lh_landmark = results.left_hand_landmarks.landmark[point]
                pixel_coordinates_lh_landmark = [normalized_lh_landmark.x, normalized_lh_landmark.y, normalized_lh_landmark.z]
                lh_data=np.concatenate((lh_data,pixel_coordinates_lh_landmark[0], pixel_coordinates_lh_landmark[1], pixel_coordinates_lh_landmark[2]),axis=None)        

        # Right hand
        if results.right_hand_landmarks is None:
            rh_data=np.concatenate((rh_data,np.zeros((rh_len_landmarks,),dtype=int)),axis=None)
        elif results.right_hand_landmarks:
            mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_drawing.DrawingSpec(color=(0,0,255),thickness=2,circle_radius=1),mp_drawing.DrawingSpec(color=(150,150,255),thickness=2,circle_radius=1))
            for point in mp_holistic.HandLandmark:
                normalized_rh_landmark = results.right_hand_landmarks.landmark[point]
                pixel_coordinates_rh_landmark = [normalized_rh_landmark.x, normalized_rh_landmark.y, normalized_rh_landmark.z]
                rh_data=np.concatenate((rh_data,pixel_coordinates_rh_landmark[0], pixel_coordinates_rh_landmark[1], pixel_coordinates_rh_landmark[2]),axis=None)        
                   
        # Save the frame-by-frame images.
        out.write(image)
        cv2.imshow('MediaPipe Hands', image)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
        iter = iter+1

# Close video display.
cap.release()
out.release()
cv2.destroyAllWindows()

# Organize data for saving.
rh_data.shape = (iter,rh_len_landmarks)
lh_data.shape = (iter,lh_len_landmarks)
body_data.shape = (iter,body_len_landmarks)

hand_data = np.concatenate((lh_data, rh_data), axis=1)

# Save hand and body data into separate .csv files.
with open(hand_output_csv_path,'a',newline='') as csvfile:
    np.savetxt(csvfile,hand_data,fmt='%f',delimiter=',')

with open(body_output_csv_path,'a',newline='') as csvfile2:
    np.savetxt(csvfile2,body_data,fmt='%f',delimiter=',')
    
    


If you want to stop the analysis at any time, press "q" for quit
Tracking complete, 327 frames analyzed
This analysis took 22.77207612991333 seconds
Your tracked output video has been saved to C:\Users\BurchK\Documents\Mediapipe_Finger_Individuation_Code\fingerIndiv_tracked.avi
Your hand output .csv data file has been saved to C:\Users\BurchK\Documents\Mediapipe_Finger_Individuation_Code\fingerIndiv_hand_outputs.csv
Your body output .csv data file has been saved to C:\Users\BurchK\Documents\Mediapipe_Finger_Individuation_Code\fingerIndiv_body_outputs.csv
